# Chapter 3: RNN-based Performance Map Prediction

This chapter explores the application of Recurrent Neural Networks (RNNs) and attention mechanisms for predicting performance maps of electric motors. We focus on efficiency and power factor maps, which are critical tools for electric motor design in hybrid and electric vehicles.

## Learning Objectives

After completing this chapter, you will understand:

- **Performance Maps**: Efficiency and power factor prediction across operating conditions
- **RNN Architectures**: GRU-based models and attention mechanisms
- **Sequence Modeling**: Handling variable-length operating point sequences
- **Transfer Learning**: Knowledge transfer between motor types and performance metrics
- **Uncertainty Quantification**: Confidence estimation for engineering applications
- **Design Optimization**: Real-time performance prediction for motor design

## Chapter Outline

1. **Motivation and Background**: Climate change, electric vehicles, and computational challenges
2. **Electric Motor Fundamentals**: Motor types, control strategies, and performance characteristics
3. **Data Generation and Processing**: Design space exploration and data representation
4. **RNN Architecture Fundamentals**: Sequence modeling and memory mechanisms
5. **Attention Mechanisms**: Encoder-decoder architectures and context vectors
6. **Model Implementation**: Modular and end-to-end approaches
7. **Transfer Learning**: Internal and external knowledge transfer
8. **Uncertainty Quantification**: Monte Carlo Dropout and confidence estimation
9. **Results and Analysis**: Performance evaluation and visualization
10. **Advanced Applications**: Design optimization and real-time prediction

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
from pathlib import Path
import json
import yaml
from datetime import datetime
import logging

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Configure TensorFlow
tf.config.experimental.enable_memory_growth = True
tf.get_logger().setLevel('ERROR')

print("🚀 Chapter 3: RNN-based Performance Map Prediction")
print("=" * 60)
print(f"📦 TensorFlow version: {tf.__version__}")
print(f"📦 PyTorch version: {torch.__version__}")
print(f"📦 NumPy version: {np.__version__}")
print(f"📦 Matplotlib version: {plt.matplotlib.__version__}")
print("✅ All imports successful!")

## Section 1: Motivation and Background

### Climate Change and Electric Vehicle Adoption

The Paris Climate Change Conference (COP 21) established ambitious targets to limit global warming to well below 2°C compared to pre-industrial levels. The transportation sector represents one of the largest contributors to greenhouse gas emissions, accounting for **28% of total U.S. GHG emissions** in 2018.

#### Key Statistics:
- **Global CO₂ emissions**: 14% from transportation sector
- **U.S. transportation**: 28% of national GHG emissions
- **Light-duty vehicles**: 59% of transportation emissions
- **Medium/heavy trucks**: 23% of transportation emissions

Electric vehicles (EVs) offer a promising solution to reduce transportation emissions. Studies show that EVs cost **less than half as much to operate** as gas-powered cars, making them increasingly attractive for consumers.

In [ ]:
# Visualize climate change and transportation emissions data
def visualize_climate_context():
    """Create visualizations for climate change context and EV adoption"""
    
    # Create sample data for demonstration
    years = np.arange(1990, 2024)
    
    # Simulated CO2 emissions data (GtCO2/year)
    global_co2 = 22.5 + 0.15 * (years - 1990) + 0.02 * (years - 1990)**2 / 10
    
    # Simulated temperature anomaly data
    temp_anomaly = 0.3 + 0.018 * (years - 1990) + 0.0001 * (years - 1990)**2
    
    # U.S. transportation sector emissions
    transport_sectors = ['Light-duty vehicles', 'Medium/heavy trucks', 
                        'Aircraft', 'Rail', 'Ships', 'Other']
    emissions_percent = [59, 23, 9, 2, 3, 4]
    
    # EV adoption data
    ev_years = np.arange(2010, 2024)
    ev_sales = np.array([0.01, 0.02, 0.05, 0.1, 0.16, 0.18, 0.20, 0.24, 
                       0.29, 0.36, 0.43, 0.66, 1.4, 1.6])  # Million units
    
    # Create comprehensive visualization
    fig = plt.figure(figsize=(20, 12))
    
    # Plot 1: Global CO2 emissions and temperature anomaly
    ax1 = plt.subplot(2, 3, (1, 2))
    ax1_twin = ax1.twinx()
    
    line1 = ax1.plot(years, global_co2, 'r-', linewidth=3, label='Global CO₂ Emissions')
    line2 = ax1_twin.plot(years, temp_anomaly, 'b-', linewidth=3, label='Temperature Anomaly')
    
    ax1.set_xlabel('Year', fontsize=12, fontweight='bold')
    ax1.set_ylabel('CO₂ Emissions (GtCO₂/year)', color='r', fontsize=12, fontweight='bold')
    ax1_twin.set_ylabel('Temperature Anomaly (°C)', color='b', fontsize=12, fontweight='bold')
    ax1.set_title('Global CO₂ Emissions and Temperature Rise', fontsize=14, fontweight='bold')
    
    # Add Paris Agreement target line
    ax1_twin.axhline(y=2.0, color='g', linestyle='--', linewidth=2, alpha=0.7, label='Paris Agreement Target')
    
    # Combine legends
    lines = line1 + line2 + [ax1_twin.lines[-1]]
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    
    # Plot 2: U.S. transportation sector breakdown
    ax2 = plt.subplot(2, 3, 3)
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
    wedges, texts, autotexts = ax2.pie(emissions_percent, labels=transport_sectors, 
                                      autopct='%1.1f%%', colors=colors, startangle=90,
                                      textprops={'fontsize': 10})
    
    # Enhance text visibility
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    
    ax2.set_title('U.S. Transportation\nSector Emissions', fontsize=12, fontweight='bold')
    
    # Plot 3: EV adoption trend
    ax3 = plt.subplot(2, 3, (4, 5))
    bars = ax3.bar(ev_years, ev_sales, color='#2ECC71', alpha=0.7)
    
    # Add value labels on bars
    for bar, value in zip(bars, ev_sales):
        if value > 0.1:  # Only show labels for visible values
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
                    f'{value:.1f}M', ha='center', va='bottom', fontweight='bold')
    
    ax3.set_xlabel('Year', fontsize=12, fontweight='bold')
    ax3.set_ylabel('EV Sales (Million Units)', fontsize=12, fontweight='bold')
    ax3.set_title('Global Electric Vehicle Sales Growth', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Add exponential trend line
    from scipy.optimize import curve_fit
    def exp_func(x, a, b):
        return a * np.exp(b * x)
    
    try:
        x_fit = ev_years - ev_years[0]
        popt, _ = curve_fit(exp_func, x_fit, ev_sales, maxfev=10000)
        x_smooth = np.linspace(0, x_fit[-1], 100)
        y_fit = exp_func(x_smooth, *popt)
        ax3.plot(ev_years[0] + x_smooth, y_fit, 'r--', linewidth=2, 
                label=f'Exponential Trend (y={popt[0]:.3f}e^{{{popt[1]:.3f}x}})', alpha=0.7)
        ax3.legend()
    except:
        pass
    
    # Plot 4: EV cost comparison
    ax4 = plt.subplot(2, 3, 6)
    
    vehicle_types = ['Gas-powered\ncar', 'Hybrid\nvehicle', 'Electric\nvehicle']
    operating_costs = [1500, 900, 600]  # Annual operating cost in USD
    colors_cost = ['#E74C3C', '#F39C12', '#27AE60']
    
    bars = ax4.bar(vehicle_types, operating_costs, color=colors_cost, alpha=0.7)
    
    # Add value labels and percentage savings
    base_cost = operating_costs[0]
    for i, (bar, cost) in enumerate(zip(bars, operating_costs)):
        savings = ((base_cost - cost) / base_cost) * 100
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                f'${cost}\n({savings:.0f}% savings)', ha='center', va='bottom', 
                fontweight='bold', fontsize=10)
    
    ax4.set_ylabel('Annual Operating Cost (USD)', fontsize=12, fontweight='bold')
    ax4.set_title('Vehicle Operating Cost Comparison', fontsize=12, fontweight='bold')
    ax4.set_ylim(0, 2000)
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Climate Change Context and Electric Vehicle Adoption', 
                fontsize=16, fontweight='bold', y=0.95)
    plt.tight_layout()
    plt.show()
    
    # Print key insights
    print("🌍 Climate Change and EV Adoption Insights:")
    print("=" * 50)
    print(f"📈 Global CO₂ emissions have increased by {((global_co2[-1] - global_co2[0]) / global_co2[0] * 100):.1f}% since 1990")
    print(f"🌡️ Global temperature has risen by {temp_anomaly[-1]:.2f}°C since 1990")
    print(f"🚗 Light-duty vehicles account for {emissions_percent[0]}% of U.S. transportation emissions")
    print(f"⚡ EV sales have grown by {((ev_sales[-1] - ev_sales[0]) / ev_sales[0] * 100):.0f}% since 2010")
    print(f"💰 EV owners save ${operating_costs[0] - operating_costs[2]:.0f} annually compared to gas-powered cars")

visualize_climate_context()

### The Need for Efficient Electric Motors

Electric motors are the **primary components** of electric drivetrains. Unlike industrial motors that operate at fixed points, EV motors must operate efficiently across the **entire torque-speed range** to:

1. **Maximize driving range** through high efficiency
2. **Provide optimal performance** during acceleration and cruising
3. **Ensure thermal stability** under varying load conditions
4. **Meet safety standards** under all operating conditions

### Performance Maps: Critical Design Tools

A **performance map** is a 2D representation showing motor performance (efficiency or power factor) across different torque-speed operating points. These maps are essential for:

- **Vehicle simulation**: Predicting range and energy consumption
- **Control strategy optimization**: Finding optimal operating points
- **Design optimization**: Balancing competing performance requirements
- **Thermal management**: Identifying high-loss operating regions

### Computational Challenges

Traditional methods for generating performance maps rely on **Finite Element (FE) analysis**, which presents significant challenges:

- **Computational cost**: 1-2 hours per complete map
- **Design space exploration**: Months for comprehensive studies
- **Real-time applications**: Not feasible for on-board prediction
- **Optimization loops**: Computationally prohibitive for iterative design

In [ ]:
# Demonstrate performance map concepts and computational challenges
def demonstrate_performance_maps():
    """Visualize performance map concepts and computational challenges"""
    
    # Create sample efficiency map data
    speed_points = np.linspace(0, 6000, 50)  # RPM
    torque_points = np.linspace(0, 250, 40)   # Nm
    
    # Create meshgrid for 2D performance map
    SPEED, TORQUE = np.meshgrid(speed_points, torque_points)
    
    # Simulate realistic efficiency map
    def efficiency_model(speed, torque):
        """Realistic efficiency map model"""
        # Base efficiency with speed and torque dependence
        base_eff = 0.85 + 0.10 * np.exp(-((speed - 3000)**2) / (2 * 1500**2))
        
        # Torque efficiency factor
        torque_factor = 1.0 - 0.3 * np.exp(-((torque - 150)**2) / (2 * 80**2))
        
        # Combined efficiency
        efficiency = base_eff * torque_factor
        
        # Add some realistic variation
        noise = 0.02 * np.random.randn(*speed.shape)
        efficiency += noise
        
        # Ensure physical limits
        efficiency = np.clip(efficiency, 0.0, 0.95)
        
        return efficiency
    
    # Generate efficiency map
    efficiency_map = efficiency_model(SPEED, TORQUE)
    
    # Create power factor map (typically correlates with efficiency)
    power_factor_map = 0.7 + 0.25 * efficiency_map + 0.05 * np.random.randn(*efficiency_map.shape)
    power_factor_map = np.clip(power_factor_map, 0.0, 1.0)
    
    # Create comprehensive visualization
    fig = plt.figure(figsize=(20, 10))
    
    # Plot 1: Efficiency Map
    ax1 = plt.subplot(2, 4, 1)
    im1 = ax1.contourf(SPEED, TORQUE, efficiency_map, levels=20, cmap='viridis')
    ax1.set_xlabel('Speed (RPM)', fontweight='bold')
    ax1.set_ylabel('Torque (Nm)', fontweight='bold')
    ax1.set_title('Efficiency Map', fontsize=14, fontweight='bold')
    plt.colorbar(im1, ax=ax1, label='Efficiency')
    
    # Add operating region contours
    efficiency_contours = ax1.contour(SPEED, TORQUE, efficiency_map, 
                                    levels=[0.8, 0.85, 0.9], colors='white', linewidths=2)
    ax1.clabel(efficiency_contours, inline=True, fontsize=10, fmt='%.2f')
    
    # Plot 2: Power Factor Map
    ax2 = plt.subplot(2, 4, 2)
    im2 = ax2.contourf(SPEED, TORQUE, power_factor_map, levels=20, cmap='plasma')
    ax2.set_xlabel('Speed (RPM)', fontweight='bold')
    ax2.set_ylabel('Torque (Nm)', fontweight='bold')
    ax2.set_title('Power Factor Map', fontsize=14, fontweight='bold')
    plt.colorbar(im2, ax=ax2, label='Power Factor')
    
    # Plot 3: Operating Points Distribution
    ax3 = plt.subplot(2, 4, 3)
    
    # Simulate typical driving cycle operating points
    n_points = 500
    driving_speeds = np.random.normal(3000, 1000, n_points)
    driving_speeds = np.clip(driving_speeds, 500, 5500)
    
    driving_torques = np.random.normal(100, 50, n_points)
    driving_torques = np.clip(driving_torques, 10, 200)
    
    scatter = ax3.scatter(driving_speeds, driving_torques, c=efficiency_model(driving_speeds, driving_torques), 
                        cmap='viridis', alpha=0.6, s=20)
    ax3.set_xlabel('Speed (RPM)', fontweight='bold')
    ax3.set_ylabel('Torque (Nm)', fontweight='bold')
    ax3.set_title('Typical Driving Cycle\nOperating Points', fontsize=14, fontweight='bold')
    plt.colorbar(scatter, ax=ax3, label='Efficiency')
    
    # Plot 4: Computational Time Comparison
    ax4 = plt.subplot(2, 4, 4)
    
    methods = ['FE Analysis', 'Traditional ML', 'Deep Learning (CNN)', 'Deep Learning (RNN)']
    times = [7200, 300, 15, 5]  # seconds per map
    colors_time = ['#E74C3C', '#F39C12', '#3498DB', '#2ECC71']
    
    bars = ax4.bar(methods, times, color=colors_time, alpha=0.7)
    ax4.set_ylabel('Time (seconds)', fontweight='bold')
    ax4.set_title('Computation Time per Map', fontsize=14, fontweight='bold')
    ax4.set_yscale('log')
    
    # Add time labels
    for bar, time in zip(bars, times):
        if time < 60:
            label = f'{time}s'
        else:
            hours = time // 3600
            minutes = (time % 3600) // 60
            if hours > 0:
                label = f'{hours}h {minutes}m'
            else:
                label = f'{minutes}m'
        
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1, 
                label, ha='center', va='bottom', fontweight='bold')
    
    plt.setp(ax4.get_xticklabels(), rotation=45, ha='right')
    
    # Plot 5: Efficiency Distribution
    ax5 = plt.subplot(2, 4, 5)
    efficiency_flat = efficiency_map.flatten()
    ax5.hist(efficiency_flat, bins=30, color='green', alpha=0.7, edgecolor='black')
    ax5.axvline(np.mean(efficiency_flat), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {np.mean(efficiency_flat):.3f}')
    ax5.set_xlabel('Efficiency', fontweight='bold')
    ax5.set_ylabel('Frequency', fontweight='bold')
    ax5.set_title('Efficiency Distribution', fontsize=14, fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Plot 6: High-Efficiency Regions
    ax6 = plt.subplot(2, 4, 6)
    high_eff_mask = efficiency_map > 0.85
    im6 = ax6.contourf(SPEED, TORQUE, high_eff_mask, levels=[0.5, 1.5], colors=['lightgreen', 'darkgreen'])
    ax6.set_xlabel('Speed (RPM)', fontweight='bold')
    ax6.set_ylabel('Torque (Nm)', fontweight='bold')
    ax6.set_title('High-Efficiency Regions\n(>85%)', fontsize=14, fontweight='bold')
    
    # Add percentage of high-efficiency area
    high_eff_percentage = np.sum(high_eff_mask) / high_eff_mask.size * 100
    ax6.text(0.05, 0.95, f'{high_eff_percentage:.1f}% of\noperating area', 
            transform=ax6.transAxes, fontweight='bold', fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", facecolor='yellow', alpha=0.7))
    
    # Plot 7: Design Optimization Challenge
    ax7 = plt.subplot(2, 4, 7)
    
    # Show design space as a 2D scatter plot
    n_designs = 100
    param1 = np.random.uniform(0.5, 1.5, n_designs)  # Normalized design parameter 1
    param2 = np.random.uniform(0.5, 1.5, n_designs)  # Normalized design parameter 2
    
    # Simulate performance based on design parameters
    performance = 0.75 + 0.15 * np.exp(-((param1 - 1.0)**2 + (param2 - 1.0)**2) / 0.5) + 0.05 * np.random.randn(n_designs)
    performance = np.clip(performance, 0.6, 0.95)
    
    scatter7 = ax7.scatter(param1, param2, c=performance, cmap='viridis', s=50, alpha=0.7)
    ax7.set_xlabel('Design Parameter 1 (normalized)', fontweight='bold')
    ax7.set_ylabel('Design Parameter 2 (normalized)', fontweight='bold')
    ax7.set_title('Design Space Exploration', fontsize=14, fontweight='bold')
    plt.colorbar(scatter7, ax=ax7, label='Peak Efficiency')
    
    # Mark optimal design
    optimal_idx = np.argmax(performance)
    ax7.scatter(param1[optimal_idx], param2[optimal_idx], c='red', s=200, 
               marker='*', edgecolors='black', linewidth=2, label='Optimal Design')
    ax7.legend()
    
    # Plot 8: RNN Approach Benefits
    ax8 = plt.subplot(2, 4, 8)
    
    benefits = ['Speed', 'Accuracy', 'Scalability', 'Flexibility']
    rnn_scores = [9, 8, 9, 8]  # Out of 10
    fe_scores = [2, 9, 3, 4]    # Out of 10
    
    x = np.arange(len(benefits))
    width = 0.35
    
    bars1 = ax8.bar(x - width/2, rnn_scores, width, label='RNN Approach', color='#2ECC71', alpha=0.7)
    bars2 = ax8.bar(x + width/2, fe_scores, width, label='FE Analysis', color='#E74C3C', alpha=0.7)
    
    ax8.set_xlabel('Evaluation Criteria', fontweight='bold')
    ax8.set_ylabel('Score (1-10)', fontweight='bold')
    ax8.set_title('RNN vs FE Analysis', fontsize=14, fontweight='bold')
    ax8.set_xticks(x)
    ax8.set_xticklabels(benefits)
    ax8.legend()
    ax8.set_ylim(0, 10)
    ax8.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax8.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                    f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
    plt.suptitle('Performance Maps: Concepts and Computational Challenges', 
                fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
    
    # Print key insights
    print("⚡ Performance Map Insights:")
    print("=" * 40)
    print(f"📊 Average efficiency across operating range: {np.mean(efficiency_flat):.3f}")
    print(f"🎯 High-efficiency operating area: {high_eff_percentage:.1f}%")
    print(f"⏱️ FE analysis time: {times[0]//3600}h {times[0]%3600//60}m per map")
    print(f"🚀 RNN prediction time: {times[3]}s per map ({times[0]//times[3]}x faster)")
    print(f"🔧 Peak design efficiency: {performance[optimal_idx]:.3f}")
    print(f"💡 RNN approach provides {times[0]//times[3]}x speedup with comparable accuracy")

demonstrate_performance_maps()

## Section 2: Electric Motor Fundamentals

### Motor Types and Geometries

This work focuses on two main types of synchronous permanent magnet motors:

#### 1. Interior Permanent Magnet (IPM) Motor
- **Configuration**: 24 slots, 4 poles
- **Application**: High-performance EV applications
- **Features**: High torque density, good flux-weakening capability
- **Design parameters**: 12 geometric variables (X₁-X₁₂)

#### 2. Fractional-Slot Concentrated Winding (FSCW) PM Motor
- **Configuration**: 12 slots, 10 poles
- **Application**: Compact EV applications
- **Features**: High efficiency, low cogging torque
- **Design parameters**: 10 geometric variables (X₁-X₁₀)

### Performance Characteristics

#### Control Strategies
1. **Maximum Torque Per Ampere (MTPA)**: Optimize torque production per unit current
2. **Flux Weakening**: Extend speed range beyond base speed
3. **Maximum Torque Per Volt (MTPV)**: Optimize torque under voltage constraints

#### Performance Maps
- **Efficiency Map**: η(N, T) - efficiency as function of speed and torque
- **Power Factor Map**: pf(N, T) - power factor as function of speed and torque
- **Loss Maps**: Copper and iron losses distribution
- **Thermal Maps**: Temperature distribution under operating conditions

In [ ]:
# Motor geometry and characteristics visualization
def visualize_motor_types():
    """Visualize IPM and FSCW motor geometries and characteristics"""
    
    # Create motor geometry visualizations
    fig = plt.figure(figsize=(20, 12))
    
    # IPM Motor Geometry
    ax1 = plt.subplot(2, 4, 1)
    
    # Simplified IPM motor cross-section
    theta = np.linspace(0, 2*np.pi, 100)
    
    # Stator outline
    stator_outer = 100
    stator_inner = 60
    rotor_outer = 55
    
    # Draw stator
    ax1.plot(stator_outer * np.cos(theta), stator_outer * np.sin(theta), 'k-', linewidth=2)
    ax1.plot(stator_inner * np.cos(theta), stator_inner * np.sin(theta), 'k-', linewidth=2)
    
    # Draw rotor
    ax1.fill_between(rotor_outer * np.cos(theta), rotor_outer * np.sin(theta), 
                    color='lightgray', alpha=0.5)
    
    # Draw simplified magnets (V-shaped)
    magnet_angles = np.array([0, 45, 90, 135]) * np.pi / 180
    for angle in magnet_angles:
        # V-shaped magnet representation
        mag_x1 = [35 * np.cos(angle - 0.2), 25 * np.cos(angle), 35 * np.cos(angle + 0.2)]
        mag_y1 = [35 * np.sin(angle - 0.2), 25 * np.sin(angle), 35 * np.sin(angle + 0.2)]
        ax1.fill(mag_x1, mag_y1, color='red', alpha=0.7)
        
        mag_x2 = [-35 * np.cos(angle - 0.2), -25 * np.cos(angle), -35 * np.cos(angle + 0.2)]
        mag_y2 = [-35 * np.sin(angle - 0.2), -25 * np.sin(angle), -35 * np.sin(angle + 0.2)]
        ax1.fill(mag_x2, mag_y2, color='blue', alpha=0.7)
    
    # Draw slots
    n_slots = 24
    slot_angles = np.linspace(0, 2*np.pi, n_slots, endpoint=False)
    for angle in slot_angles:
        slot_x = [stator_inner * np.cos(angle), stator_outer * np.cos(angle)]
        slot_y = [stator_inner * np.sin(angle), stator_outer * np.sin(angle)]
        ax1.plot(slot_x, slot_y, 'k-', linewidth=1)
    
    ax1.set_xlim(-120, 120)
    ax1.set_ylim(-120, 120)
    ax1.set_aspect('equal')
    ax1.set_title('IPM Motor\n(24 slots, 4 poles)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('x (mm)')
    ax1.set_ylabel('y (mm)')
    ax1.grid(True, alpha=0.3)
    
    # FSCW Motor Geometry
    ax2 = plt.subplot(2, 4, 2)
    
    # Simplified FSCW motor cross-section
    stator_outer = 100
    stator_inner = 60
    rotor_outer = 55
    
    # Draw stator
    ax2.plot(stator_outer * np.cos(theta), stator_outer * np.sin(theta), 'k-', linewidth=2)
    ax2.plot(stator_inner * np.cos(theta), stator_inner * np.sin(theta), 'k-', linewidth=2)
    
    # Draw rotor
    ax2.fill_between(rotor_outer * np.cos(theta), rotor_outer * np.sin(theta), 
                    color='lightgray', alpha=0.5)
    
    # Draw surface magnets
    n_poles = 10
    pole_angles = np.linspace(0, 2*np.pi, n_poles, endpoint=False)
    for i, angle in enumerate(pole_angles):
        color = 'red' if i % 2 == 0 else 'blue'
        pole_x1 = rotor_outer * np.cos(angle - np.pi/n_poles/2)
        pole_y1 = rotor_outer * np.sin(angle - np.pi/n_poles/2)
        pole_x2 = rotor_outer * np.cos(angle + np.pi/n_pole/2)
        pole_y2 = rotor_outer * np.sin(angle + np.pi/n_pole/2)
        
        inner_x = 45 * np.cos(angle)
        inner_y = 45 * np.sin(angle)
        
        ax2.fill([pole_x1, pole_x2, inner_x], [pole_y1, pole_y2, inner_y], 
                color=color, alpha=0.7)
    
    # Draw concentrated slots
    n_slots = 12
    slot_angles = np.linspace(0, 2*np.pi, n_slots, endpoint=False)
    for angle in slot_angles:
        # Wider slot opening for concentrated winding
        slot_width = 0.15  # radians
        slot_x1 = [stator_inner * np.cos(angle - slot_width/2), 
                  stator_outer * np.cos(angle - slot_width/2)]
        slot_y1 = [stator_inner * np.sin(angle - slot_width/2), 
                  stator_outer * np.sin(angle - slot_width/2)]
        slot_x2 = [stator_inner * np.cos(angle + slot_width/2), 
                  stator_outer * np.cos(angle + slot_width/2)]
        slot_y2 = [stator_inner * np.sin(angle + slot_width/2), 
                  stator_outer * np.sin(angle + slot_width/2)]
        
        ax2.plot([slot_x1[0], slot_x2[0]], [slot_y1[0], slot_y2[0]], 'k-', linewidth=2)
        ax2.plot(slot_x1, slot_y1, 'k-', linewidth=1)
        ax2.plot(slot_x2, slot_y2, 'k-', linewidth=1)
    
    ax2.set_xlim(-120, 120)
    ax2.set_ylim(-120, 120)
    ax2.set_aspect('equal')
    ax2.set_title('FSCW PM Motor\n(12 slots, 10 poles)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('x (mm)')
    ax2.set_ylabel('y (mm)')
    ax2.grid(True, alpha=0.3)
    
    # Design Parameter Comparison
    ax3 = plt.subplot(2, 4, 3)
    
    # Create parameter comparison table as text
    parameter_data = [
        ['Parameter', 'IPM Motor', 'FSCW Motor'],
        ['Slots', '24', '12'],
        ['Poles', '4', '10'],
        ['Rated Current', '33.33 A', '300 A'],
        ['DC Voltage', '400 V', '450 V'],
        ['Design Vars', '12 (X₁-X₁₂)', '10 (X₁-X₁₀)']
    ]
    
    table_data = []
    for row in parameter_data[1:]:
        table_data.append(row)
    
    table = ax3.table(cellText=table_data,
                    colLabels=parameter_data[0],
                    cellLoc='center',
                    loc='center',
                    colWidths=[0.4, 0.3, 0.3])
    
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    # Style the table
    for i in range(len(parameter_data)):
        for j in range(len(parameter_data[0])):
            cell = table[(i, j)]
            if i == 0:  # Header row
                cell.set_facecolor('#4CAF50')
                cell.set_text_props(weight='bold', color='white')
            else:
                cell.set_facecolor('#f0f0f0' if i % 2 == 0 else 'white')
    
    ax3.axis('off')
    ax3.set_title('Motor Specifications', fontsize=12, fontweight='bold')
    
    # Performance Characteristics Comparison
    ax4 = plt.subplot(2, 4, 4)
    
    performance_metrics = ['Torque\nDensity', 'Efficiency', 'Power\nFactor', 'Cogging\nTorque', 'Speed\nRange']
    ipm_scores = [8, 7, 8, 6, 9]
    fscw_scores = [7, 9, 7, 9, 8]
    
    x = np.arange(len(performance_metrics))
    width = 0.35
    
    bars1 = ax4.bar(x - width/2, ipm_scores, width, label='IPM', color='#3498DB', alpha=0.7)
    bars2 = ax4.bar(x + width/2, fscw_scores, width, label='FSCW', color='#E74C3C', alpha=0.7)
    
    ax4.set_xlabel('Performance Metrics', fontweight='bold')
    ax4.set_ylabel('Score (1-10)', fontweight='bold')
    ax4.set_title('Performance Comparison', fontsize=12, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels(performance_metrics)
    ax4.legend()
    ax4.set_ylim(0, 10)
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Control Strategy Regions
    ax5 = plt.subplot(2, 4, 5)
    
    # Create operating regions visualization
    speed = np.linspace(0, 6000, 100)
    torque = np.linspace(0, 250, 100)
    SPEED, TORQUE = np.meshgrid(speed, torque)
    
    # Define operating regions
    base_speed = 3000
    
    # MTPA region (below base speed)
    mtpa_region = SPEED <= base_speed
    
    # Flux weakening region (above base speed)
    fw_region = SPEED > base_speed
    
    # Create operating region map
    region_map = np.zeros_like(SPEED)
    region_map[mtpa_region] = 1  # MTPA
    region_map[fw_region] = 2    # Flux weakening
    
    # Add torque limit envelope
    torque_limit = 200 * np.exp(-SPEED / 8000)  # Exponential torque drop
    for i in range(len(torque)):
        for j in range(len(speed)):
            if TORQUE[i, j] > torque_limit[j]:
                region_map[i, j] = 0  # Outside operating envelope
    
    im5 = ax5.contourf(SPEED, TORQUE, region_map, levels=[0, 1, 2, 3], 
                     colors=['lightgray', 'lightblue', 'lightcoral'], alpha=0.7)
    
    ax5.set_xlabel('Speed (RPM)', fontweight='bold')
    ax5.set_ylabel('Torque (Nm)', fontweight='bold')
    ax5.set_title('Control Strategy Regions', fontsize=12, fontweight='bold')
    
    # Add region labels
    ax5.text(1500, 200, 'MTPA Region', fontsize=12, fontweight='bold', 
            ha='center', va='center', bbox=dict(boxstyle="round,pad=0.3", facecolor='lightblue', alpha=0.8))
    ax5.text(4500, 100, 'Flux Weakening\nRegion', fontsize=12, fontweight='bold', 
            ha='center', va='center', bbox=dict(boxstyle="round,pad=0.3", facecolor='lightcoral', alpha=0.8))
    ax5.text(3000, 30, f'Base Speed\n{base_speed} RPM', fontsize=10, fontweight='bold', 
            ha='center', va='center', bbox=dict(boxstyle="round,pad=0.3", facecolor='yellow', alpha=0.8))
    
    # Draw torque limit line
    ax5.plot(speed, torque_limit, 'k-', linewidth=2, label='Torque Limit')
    ax5.axvline(x=base_speed, color='green', linestyle='--', linewidth=2, label='Base Speed')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Efficiency Maps for Both Motors
    ax6 = plt.subplot(2, 4, 6)
    
    # Create sample efficiency maps for both motors
    def create_efficiency_map(center_speed, spread_speed, center_torque, spread_torque):
        efficiency = 0.75 + 0.15 * np.exp(-((SPEED - center_speed)**2) / (2 * spread_speed**2) 
                                     - ((TORQUE - center_torque)**2) / (2 * spread_torque**2))
        efficiency += 0.02 * np.random.randn(*efficiency.shape)
        return np.clip(efficiency, 0.6, 0.95)
    
    ipm_efficiency = create_efficiency_map(3000, 1500, 150, 80)
    fscw_efficiency = create_efficiency_map(2500, 1200, 120, 60)
    
    # Plot IPM efficiency
    im6 = ax6.contourf(SPEED, TORQUE, ipm_efficiency, levels=15, cmap='viridis', alpha=0.7)
    ax6.set_xlabel('Speed (RPM)', fontweight='bold')
    ax6.set_ylabel('Torque (Nm)', fontweight='bold')
    ax6.set_title('IPM Efficiency Map', fontsize=12, fontweight='bold')
    plt.colorbar(im6, ax=ax6, label='Efficiency')
    
    # Power Factor Maps for Both Motors
    ax7 = plt.subplot(2, 4, 7)
    
    ipm_pf = 0.7 + 0.25 * ipm_efficiency + 0.05 * np.random.randn(*ipm_efficiency.shape)
    ipm_pf = np.clip(ipm_pf, 0.5, 1.0)
    
    im7 = ax7.contourf(SPEED, TORQUE, ipm_pf, levels=15, cmap='plasma', alpha=0.7)
    ax7.set_xlabel('Speed (RPM)', fontweight='bold')
    ax7.set_ylabel('Torque (Nm)', fontweight='bold')
    ax7.set_title('IPM Power Factor Map', fontsize=12, fontweight='bold')
    plt.colorbar(im7, ax=ax7, label='Power Factor')
    
    # Design Parameter Sensitivity
    ax8 = plt.subplot(2, 4, 8)
    
    # Show sensitivity of key design parameters
    params = ['Magnet\nWidth', 'Air Gap', 'Slot\nDepth', 'Core\nLength']
    ipm_sensitivity = [0.8, 0.6, 0.4, 0.7]
    fscw_sensitivity = [0.6, 0.8, 0.7, 0.5]
    
    x = np.arange(len(params))
    width = 0.35
    
    bars1 = ax8.bar(x - width/2, ipm_sensitivity, width, label='IPM', color='#3498DB', alpha=0.7)
    bars2 = ax8.bar(x + width/2, fscw_sensitivity, width, label='FSCW', color='#E74C3C', alpha=0.7)
    
    ax8.set_xlabel('Design Parameters', fontweight='bold')
    ax8.set_ylabel('Sensitivity', fontweight='bold')
    ax8.set_title('Parameter Sensitivity', fontsize=12, fontweight='bold')
    ax8.set_xticks(x)
    ax8.set_xticklabels(params)
    ax8.legend()
    ax8.set_ylim(0, 1)
    ax8.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Electric Motor Types and Characteristics', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
    
    # Print key insights
    print("⚡ Electric Motor Characteristics:")
    print("=" * 40)
    print(f"🔧 IPM Motor: {parameter_data[1][1]} slots, {parameter_data[2][1]} poles, {parameter_data[5][1]} design variables")
    print(f"🔧 FSCW Motor: {parameter_data[1][2]} slots, {parameter_data[2][2]} poles, {parameter_data[5][2]} design variables")
    print(f"📊 IPM Peak Torque Density Score: {max(ipm_scores)}/10")
    print(f"📊 FSCW Peak Efficiency Score: {max(fscw_scores)}/10")
    print(f"⚙️ Base Speed: {base_speed} RPM (transition between control strategies)")
    print(f"🎯 High-efficiency region area: {np.sum(ipm_efficiency > 0.85) / ipm_efficiency.size * 100:.1f}% (IPM)")
    print(f"🔍 Most sensitive parameter (IPM): {params[np.argmax(ipm_sensitivity)]} ({ipm_sensitivity[np.argmax(ipm_sensitivity)]:.1f})")

visualize_motor_types()

## Section 3: Data Generation and Processing

### Design Space Exploration

#### Latin Hypercube Sampling (LHS)
**Latin Hypercube Sampling** is used to efficiently explore the multi-dimensional design space:

- **Stratified sampling**: Ensures uniform coverage of parameter space
- **Dimensional independence**: Each parameter is independently sampled
- **Efficiency**: Better space-filling properties than random sampling
- **Reproducibility**: Controlled randomization for consistent results

For our motor design problem:
- **IPM Motor**: 12 design parameters (X₁-X₁₂) ±50% variation
- **FSCW Motor**: 10 design parameters (X₁-X₁₀) ±50% variation
- **Sample sizes**: 2900 (IPM), 3200 (FSCW) total designs
- **Data split**: 2500 training, 200 validation, 200 test

### 4-Stage Performance Map Generation

The traditional FE-based performance map generation involves 4 stages:

#### Stage 1: Flux Linkage Characterization
- **Input**: Motor geometry and material distribution
- **Process**: FE simulations at 9 d-q current points
- **Output**: d-q flux linkage maps λ_d(I_d, I_q), λ_q(I_d, I_q)
- **Method**: 2nd-degree polynomial interpolation

#### Stage 2: Control Strategy Optimization
- **Input**: Flux linkage maps from Stage 1
- **Process**: Apply control strategies (MTPA, FW, MTPV)
- **Output**: Optimal excitation conditions (I_s*, γ*) for each speed
- **Methods**: Numerical optimization for each operating point

#### Stage 3: Performance Calculation
- **Input**: Optimal excitation points and motor geometry
- **Process**: FE simulations at optimized operating points
- **Output**: Torque (T_em*), efficiency (η*), power factor values
- **Method**: Electromagnetic and loss calculations

#### Stage 4: Map Generation
- **Input**: Discrete performance points from Stage 3
- **Process**: Interpolation and post-processing
- **Output**: Continuous performance maps
- **Methods**: 2D interpolation, smoothing, contour generation

In [ ]:
# Data generation and processing implementation
class MotorDesignGenerator:
    """Generate motor design samples using Latin Hypercube Sampling"""
    
    def __init__(self, motor_type='IPM', n_samples=100):
        self.motor_type = motor_type
        self.n_samples = n_samples
        
        # Define design parameters for each motor type
        if motor_type == 'IPM':
            self.param_names = [f'X{i}' for i in range(1, 13)]
            self.param_ranges = {
                'X1': (13.5, 22.5),   # mm
                'X2': (4.35, 6.65),   # mm
                'X3': (2.25, 3.75),   # mm
                'X4': (0.75, 1.25),   # mm
                'X5': (0.75, 1.25),   # mm
                'X6': (75, 125),      # mm
                'X7': (37.5, 62.5),   # mm
                'X8': (13.5, 22.5),   # mm
                'X9': (3, 6),         # mm
                'X10': (11.25, 18.75), # mm
                'X11': (33.75, 56.25), # degrees
                'X12': (1.125, 1.875)  # mm
            }
        else:  # FSCW
            self.param_names = [f'X{i}' for i in range(1, 11)]
            self.param_ranges = {
                'X1': (9.75, 16.25),   # mm
                'X2': (18.75, 31.25),  # mm
                'X3': (3.225, 5.375),  # mm
                'X4': (5.25, 8.75),    # mm
                'X5': (0.75, 1.25),    # mm
                'X6': (86.25, 143.75), # mm
                'X7': (91.875, 153.125), # mm
                'X8': (9.375, 15.625), # mm
                'X9': (45, 75),        # mm
                'X10': (0.1875, 0.3125) # mm
            }
    
    def latin_hypercube_sampling(self):
        """Generate Latin Hypercube samples"""
        np.random.seed(42)  # For reproducibility
        
        n_params = len(self.param_names)
        samples = np.zeros((self.n_samples, n_params))
        
        for i in range(n_params):
            # Generate random permutations
            permutation = np.random.permutation(self.n_samples)
            
            # Generate uniform samples in each interval
            uniform_samples = np.random.uniform(0, 1, self.n_samples)
            
            # Calculate LHS samples
            lhs_samples = (permutation + uniform_samples) / self.n_samples
            
            # Scale to parameter ranges
            param_name = self.param_names[i]
            min_val, max_val = self.param_ranges[param_name]
            samples[:, i] = min_val + lhs_samples * (max_val - min_val)
        
        return samples
    
    def generate_design_dataframe(self):
        """Generate a pandas DataFrame with design samples"""
        samples = self.latin_hypercube_sampling()
        
        df = pd.DataFrame(samples, columns=self.param_names)
        df['motor_type'] = self.motor_type
        df['design_id'] = range(1, self.n_samples + 1)
        
        return df

class PerformanceMapSimulator:
    """Simulate the 4-stage performance map generation process"""
    
    def __init__(self):
        self.speed_points = np.linspace(0, 6000, 25)
        self.torque_points = np.linspace(0, 250, 20)
        
    def stage1_flux_linkage_characterization(self, design_params):
        """Stage 1: Generate flux linkage maps"""
        # Simplified flux linkage model
        id_points = np.linspace(-200, 200, 3)
        iq_points = np.linspace(-200, 200, 3)
        
        ID, IQ = np.meshgrid(id_points, iq_points)
        
        # Flux linkage depends on design parameters
        base_lambda_d = 0.1 + 0.05 * np.mean(design_params) / 100
        base_lambda_q = 0.05 + 0.02 * np.mean(design_params) / 100
        
        lambda_d = base_lambda_d * (1 + 0.1 * ID / 200) * (1 + 0.05 * IQ / 200)
        lambda_q = base_lambda_q * (1 + 0.05 * ID / 200) * (1 + 0.1 * IQ / 200)
        
        return {'lambda_d': lambda_d, 'lambda_q': lambda_q, 
                'id_points': id_points, 'iq_points': iq_points}
    
    def stage2_control_optimization(self, flux_data):
        """Stage 2: Optimize control strategies"""
        base_speed = 3000  # RPM
        
        operating_points = []
        
        for speed in self.speed_points:
            if speed <= base_speed:
                # MTPA region
                torque_limit = 200 * (1 - speed / 10000)
                current_angle = np.pi / 4  # 45 degrees for MTPA
                current_magnitude = torque_limit / 0.8
            else:
                # Flux weakening region
                torque_limit = 200 * np.exp(-(speed - base_speed) / 2000)
                current_angle = np.pi / 6  # 30 degrees for FW
                current_magnitude = torque_limit / (0.8 * base_speed / speed)
            
            if torque_limit > 10:  # Only include meaningful operating points
                operating_points.append({
                    'speed': speed,
                    'torque': torque_limit,
                    'current_mag': current_magnitude,
                    'current_angle': current_angle
                })
        
        return operating_points
    
    def stage3_performance_calculation(self, operating_points, design_params):
        """Stage 3: Calculate performance at optimized points"""
        performance_data = []
        
        for op_point in operating_points:
            speed = op_point['speed']
            torque = op_point['torque']
            
            # Simplified performance model
            # Efficiency depends on operating point and design
            base_efficiency = 0.85
            speed_factor = np.exp(-((speed - 3000)**2) / (2 * 1500**2))
            torque_factor = np.exp(-((torque - 150)**2) / (2 * 80**2))
            design_factor = 1 + 0.1 * (np.mean(design_params) - 100) / 100
            
            efficiency = base_efficiency + 0.1 * speed_factor * torque_factor * design_factor
            efficiency = np.clip(efficiency, 0.6, 0.95)
            
            # Power factor correlates with efficiency
            power_factor = 0.7 + 0.25 * efficiency + 0.05 * np.random.randn()
            power_factor = np.clip(power_factor, 0.5, 1.0)
            
            # Add noise for realism
            efficiency += 0.02 * np.random.randn()
            efficiency = np.clip(efficiency, 0.6, 0.95)
            
            performance_data.append({
                'speed': speed,
                'torque': torque,
                'efficiency': efficiency,
                'power_factor': power_factor,
                'current_mag': op_point['current_mag'],
                'current_angle': op_point['current_angle']
            })
        
        return performance_data
    
    def stage4_map_generation(self, performance_data):
        """Stage 4: Generate continuous performance maps"""
        if not performance_data:
            return None
        
        # Extract data
        speeds = [p['speed'] for p in performance_data]
        torques = [p['torque'] for p in performance_data]
        efficiencies = [p['efficiency'] for p in performance_data]
            power_factors = [p['power_factor'] for p in performance_data]
        
        # Create meshgrid for interpolation
        SPEED, TORQUE = np.meshgrid(self.speed_points, self.torque_points)
        
        # Interpolate to create continuous maps
        from scipy.interpolate import griddata
        
        try:
            efficiency_map = griddata(
                (speeds, torques), efficiencies, 
                (SPEED, TORQUE), method='cubic', fill_value=0.6
            )
            
            power_factor_map = griddata(
                (speeds, torques), power_factors,
                (SPEED, TORQUE), method='cubic', fill_value=0.5
            )
        except:
            # Fallback to linear interpolation
            efficiency_map = griddata(
                (speeds, torques), efficiencies,
                (SPEED, TORQUE), method='linear', fill_value=0.6
            )
            
            power_factor_map = griddata(
                (speeds, torques), power_factors,
                (SPEED, TORQUE), method='linear', fill_value=0.5
            )
        
        return {
            'efficiency_map': efficiency_map,
            'power_factor_map': power_factor_map,
            'speed_grid': SPEED,
            'torque_grid': TORQUE,
            'raw_data': performance_data
        }

# Demonstrate data generation process
def demonstrate_data_generation():
    """Demonstrate the complete data generation workflow"""
    
    print("🔧 Data Generation and Processing Demonstration")
    print("=" * 60)
    
    # Initialize generators
    ipm_generator = MotorDesignGenerator('IPM', n_samples=50)
    fscw_generator = MotorDesignGenerator('FSCW', n_samples=50)
    simulator = PerformanceMapSimulator()
    
    # Generate design samples
    print("\n📊 Step 1: Latin Hypercube Sampling")
    ipm_designs = ipm_generator.generate_design_dataframe()
    fscw_designs = fscw_generator.generate_design_dataframe()
    
    print(f"   IPM Motor: Generated {len(ipm_designs)} design samples")
    print(f"   FSCW Motor: Generated {len(fscw_designs)} design samples")
    print(f"   Design parameters: {len(ipm_generator.param_names)} (IPM), {len(fscw_generator.param_names)} (FSCW)")
    
    # Show design space coverage
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Data Generation and Processing Workflow', fontsize=16, fontweight='bold')
    
    # Plot 1: Design space visualization (first 2 parameters)
    ax1 = axes[0, 0]
    ax1.scatter(ipm_designs['X1'], ipm_designs['X2'], alpha=0.6, s=30, label='IPM', color='blue')
    ax1.scatter(fscw_designs['X1'], fscw_designs['X2'], alpha=0.6, s=30, label='FSCW', color='red')
    ax1.set_xlabel('X1 (mm)', fontweight='bold')
    ax1.set_ylabel('X2 (mm)', fontweight='bold')
    ax1.set_title('Design Space Coverage (LHS)', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Parameter distributions
    ax2 = axes[0, 1]
    for i, param in enumerate(['X1', 'X2', 'X3']):
        ax2.hist(ipm_designs[param], alpha=0.5, label=f'IPM {param}', bins=10)
    ax2.set_xlabel('Parameter Value', fontweight='bold')
    ax2.set_ylabel('Frequency', fontweight='bold')
    ax2.set_title('Parameter Distributions', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Demonstrate 4-stage process for a sample design
    sample_design = ipm_designs.iloc[0][ipm_generator.param_names].values
    
    # Stage 1: Flux linkage characterization
    print("\n🔌 Step 2: Stage 1 - Flux Linkage Characterization")
    flux_data = simulator.stage1_flux_linkage_characterization(sample_design)
    print(f"   Generated flux linkage maps: λ_d and λ_q")
    print(f"   Current grid: {len(flux_data['id_points'])}×{len(flux_data['iq_points'])} points")
    
    # Plot 3: Flux linkage maps
    ax3 = axes[0, 2]
    ID, IQ = np.meshgrid(flux_data['id_points'], flux_data['iq_points'])
    im3 = ax3.contourf(ID, IQ, flux_data['lambda_d'], levels=15, cmap='viridis')
    ax3.set_xlabel('I_d (A)', fontweight='bold')
    ax3.set_ylabel('I_q (A)', fontweight='bold')
    ax3.set_title('d-axis Flux Linkage', fontweight='bold')
    plt.colorbar(im3, ax=ax3)
    
    # Stage 2: Control optimization
    print("⚙️ Step 3: Stage 2 - Control Strategy Optimization")
    operating_points = simulator.stage2_control_optimization(flux_data)
    print(f"   Generated {len(operating_points)} optimized operating points")
    print(f"   Speed range: {min(p['speed'] for p in operating_points):.0f}-{max(p['speed'] for p in operating_points):.0f} RPM")
    print(f"   Torque range: {min(p['torque'] for p in operating_points):.0f}-{max(p['torque'] for p in operating_points):.0f} Nm")
    
    # Plot 4: Control strategy regions
    ax4 = axes[1, 0]
    speeds = [p['speed'] for p in operating_points]
    torques = [p['torque'] for p in operating_points]
    current_mags = [p['current_mag'] for p in operating_points]
    
    scatter4 = ax4.scatter(speeds, torques, c=current_mags, cmap='plasma', s=50, alpha=0.7)
    ax4.set_xlabel('Speed (RPM)', fontweight='bold')
    ax4.set_ylabel('Torque (Nm)', fontweight='bold')
    ax4.set_title('Optimized Operating Points', fontweight='bold')
    plt.colorbar(scatter4, ax=ax4, label='Current Magnitude (A)')
    ax4.grid(True, alpha=0.3)
    
    # Stage 3: Performance calculation
    print("⚡ Step 4: Stage 3 - Performance Calculation")
    performance_data = simulator.stage3_performance_calculation(operating_points, sample_design)
    print(f"   Calculated performance at {len(performance_data)} operating points")
    
    efficiencies = [p['efficiency'] for p in performance_data]
    power_factors = [p['power_factor'] for p in performance_data]
    
    print(f"   Efficiency range: {min(efficiencies):.3f}-{max(efficiencies):.3f}")
    print(f"   Power factor range: {min(power_factors):.3f}-{max(power_factors):.3f}")
    
    # Plot 5: Performance metrics
    ax5 = axes[1, 1]
    ax5.scatter(efficiencies, power_factors, alpha=0.7, s=30, color='green')
    ax5.set_xlabel('Efficiency', fontweight='bold')
    ax5.set_ylabel('Power Factor', fontweight='bold')
    ax5.set_title('Efficiency vs Power Factor', fontweight='bold')
    ax5.grid(True, alpha=0.3)
    
    # Stage 4: Map generation
    print("🗺️ Step 5: Stage 4 - Performance Map Generation")
    performance_maps = simulator.stage4_map_generation(performance_data)
    
    if performance_maps:
        print(f"   Generated continuous performance maps")
        print(f"   Map resolution: {performance_maps['efficiency_map'].shape}")
        
        # Plot 6: Final efficiency map
        ax6 = axes[1, 2]
        im6 = ax6.contourf(performance_maps['speed_grid'], 
                          performance_maps['torque_grid'],
                          performance_maps['efficiency_map'], 
                          levels=15, cmap='viridis')
        ax6.set_xlabel('Speed (RPM)', fontweight='bold')
        ax6.set_ylabel('Torque (Nm)', fontweight='bold')
        ax6.set_title('Final Efficiency Map', fontweight='bold')
        plt.colorbar(im6, ax=ax6, label='Efficiency')
        
        # Add performance metrics
        eff_map = performance_maps['efficiency_map']
        high_eff_area = np.sum(eff_map > 0.85) / eff_map.size * 100
        
        print(f"   High-efficiency area (>85%): {high_eff_area:.1f}%")
    else:
        ax6.text(0.5, 0.5, 'Map generation failed', ha='center', va='center', 
                transform=ax6.transAxes, fontsize=14, color='red')
        ax6.set_title('Final Efficiency Map', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Create data processing pipeline visualization
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    
    # Pipeline stages
    stages = ['Design\nGeneration', 'Flux\nLinkage', 'Control\nOptimization', 
             'Performance\nCalculation', 'Map\nGeneration']
    
    stage_descriptions = [
        'Latin Hypercube\nSampling\n50-100 samples',
        '9 d-q points\n2nd-degree\nPolynomial',
        'MTPA/FW/MTPV\nOptimal\nExcitation',
        'FE Simulation\nTorque, Efficiency,\nPower Factor',
        '2D Interpolation\nContinuous\nMaps'
    ]
    
    # Create pipeline visualization
    y_positions = np.arange(len(stages))
    
    # Draw boxes for each stage
    box_width = 1.5
    box_height = 0.6
    
    for i, (stage, desc) in enumerate(zip(stages, stage_descriptions)):
        # Stage box
        rect = plt.Rectangle((2, y_positions[i] - box_height/2), box_width, box_height, 
                            facecolor='lightblue', edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        # Stage text
        ax.text(2.75, y_positions[i], stage, ha='center', va='center', 
                fontsize=11, fontweight='bold')
        
        # Description text
        ax.text(5, y_positions[i], desc, ha='left', va='center', 
                fontsize=10, bbox=dict(boxstyle="round,pad=0.3", facecolor='lightyellow', alpha=0.7))
        
        # Arrows
        if i < len(stages) - 1:
            ax.arrow(3.75, y_positions[i], 0.5, 0, head_width=0.1, head_length=0.1, 
                    fc='black', ec='black')
    
    # Add computational time indicators
    time_indicators = ['Minutes', 'Minutes', 'Seconds', 'Hours', 'Minutes']
    for i, time_str in enumerate(time_indicators):
        ax.text(7, y_positions[i], f'⏱️ {time_str}', ha='left', va='center', 
                fontsize=9, style='italic', color='red')
    
    ax.set_xlim(0, 8)
    ax.set_ylim(-0.5, len(stages) - 0.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('4-Stage Performance Map Generation Pipeline', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n📋 Data Generation Summary:")
    print("=" * 40)
    print(f"🎯 Total design samples generated: {len(ipm_designs) + len(fscw_designs)}")
    print(f"📐 Design space dimensions: {len(ipm_generator.param_names)} (IPM), {len(fscw_generator.param_names)} (FSCW)")
    print(f"⚡ Operating points per design: {len(operating_points)}")
    print(f"🗺️ Map resolution: {performance_maps['efficiency_map'].shape if performance_maps else 'N/A'}")
    print(f"🔄 Complete pipeline time: ~2-3 hours per design (FE-based)")
    print(f"🚀 Deep Learning target: ~5 seconds per design")

demonstrate_data_generation()

### Sequence Data Representation

For RNN-based prediction, the 2D performance maps must be converted to 1D sequences. Several approaches are possible:

#### Sequence Formation Strategies

1. **Row-wise flattening**: Process map row by row
2. **Column-wise flattening**: Process map column by column  
3. **Spiral ordering**: Process from center outward
4. **Optimal path ordering**: Follow efficiency contours

#### Input Features for Each Operating Point

For each point in the sequence, the input features include:

- **Operating conditions**: Speed (N), Torque (T_em)
- **Excitation parameters**: Current magnitude (I_s), Current angle (γ)
- **Motor parameters**: Base speed (N_base), Design constants
- **Geometry encoding**: CNN features or parameter vector

#### Target Variables

- **Efficiency**: η ∈ [0, 1] for each operating point
- **Power factor**: pf ∈ [0, 1] for each operating point
- **Additional metrics**: Losses, temperature, stress (if available)

#### Data Normalization

Proper normalization is critical for RNN training:

- **Efficiency/Power factor**: Divide by 100 or use min-max scaling
- **Speed/Torque**: Normalize to [0, 1] based on design limits
- **Current**: Normalize by rated current
- **Angles**: Convert to radians and normalize to [-1, 1]

This data representation enables the RNN to learn spatial relationships and dependencies between operating points while handling variable sequence lengths for different motor designs.

In [ ]:
# Sequence data processing and representation
class SequenceDataProcessor:
    """Process performance maps into sequences for RNN training"""
    
    def __init__(self, sequence_type='row_wise'):
        self.sequence_type = sequence_type
        self.scalers = {}
    
    def map_to_sequence(self, performance_map, speed_grid, torque_grid):
        """Convert 2D performance map to 1D sequence"""
        if performance_map is None:
            return None, None, None
        
        # Flatten the map to sequence based on selected strategy
        if self.sequence_type == 'row_wise':
            sequence = performance_map.flatten()
            speed_seq = speed_grid.flatten()
            torque_seq = torque_grid.flatten()
        elif self.sequence_type == 'column_wise':
            sequence = performance_map.T.flatten()
            speed_seq = speed_grid.T.flatten()
            torque_seq = torque_grid.T.flatten()
        elif self.sequence_type == 'spiral':
            sequence, speed_seq, torque_seq = self._spiral_ordering(
                performance_map, speed_grid, torque_grid)
        else:
            raise ValueError(f"Unknown sequence type: {self.sequence_type}")
        
        # Remove invalid points (where performance is 0 or NaN)
        valid_mask = (sequence > 0) & (~np.isnan(sequence))
        
        return sequence[valid_mask], speed_seq[valid_mask], torque_seq[valid_mask]
    
    def _spiral_ordering(self, performance_map, speed_grid, torque_grid):
        """Create spiral ordering from center outward"""
        center_y, center_x = performance_map.shape[0] // 2, performance_map.shape[1] // 2
        
        # Create distance matrix from center
        y_coords, x_coords = np.ogrid[:performance_map.shape[0], :performance_map.shape[1]]
        distances = np.sqrt((x_coords - center_x)**2 + (y_coords - center_y)**2)
        
        # Sort by distance, with some randomness for equal distances
        flat_distances = distances.flatten()
        sort_indices = np.lexsort((np.random.randn(len(flat_distances)), flat_distances))
        
        # Apply ordering
        sequence = performance_map.flatten()[sort_indices]
        speed_seq = speed_grid.flatten()[sort_indices]
        torque_seq = torque_grid.flatten()[sort_indices]
        
        return sequence, speed_seq, torque_seq
    
    def create_input_features(self, speed_seq, torque_seq, design_params, base_speed=3000):
        """Create input features for RNN"""
        # Normalize inputs
        speed_norm = speed_seq / 6000  # Normalize to [0, 1]
        torque_norm = torque_seq / 250  # Normalize to [0, 1]
        
        # Create operating point features
        n_points = len(speed_seq)
        
        # Feature matrix for each operating point
        features = np.zeros((n_points, 14))  # 14 input features as per original design
        
        # Operating condition features (5)
        features[:, 0] = speed_norm           # Normalized speed
        features[:, 1] = torque_norm          # Normalized torque
        features[:, 2] = speed_norm * torque_norm  # Interaction term
        features[:, 3] = speed_norm**2        # Speed squared
        features[:, 4] = torque_norm**2       # Torque squared
        
        # Control-related features (3)
        features[:, 5] = np.ones(n_points)    # Base speed normalized (1.0)
        features[:, 6] = speed_norm / (base_speed / 6000)  # Speed ratio
        features[:, 7] = torque_norm / 0.8   # Torque utilization
        
        # Geometry-related features (6) - simplified
        for i in range(6):
            if i < len(design_params):
                features[:, 8 + i] = design_params[i] / 100  # Normalize geometry
            else:
                features[:, 8 + i] = 1.0  # Default value
        
        return features
    
    def normalize_targets(self, targets):
        """Normalize target values (efficiency/power factor)"""
        # Divide by 100 to get [0, 1] range
        return targets / 100
    
    def denormalize_targets(self, normalized_targets):
        """Convert back to original scale"""
        return normalized_targets * 100

# Demonstrate sequence processing
def demonstrate_sequence_processing():
    """Demonstrate sequence data processing for RNN training"""
    
    print("📊 Sequence Data Processing Demonstration")
    print("=" * 60)
    
    # Create sample performance map
    speed_points = np.linspace(0, 6000, 20)
    torque_points = np.linspace(0, 250, 15)
    SPEED, TORQUE = np.meshgrid(speed_points, torque_points)
    
    # Create realistic efficiency map
    efficiency_map = 0.75 + 0.15 * np.exp(-((SPEED - 3000)**2) / (2 * 1500**2) 
                               - ((TORQUE - 125)**2) / (2 * 60**2))
    efficiency_map += 0.02 * np.random.randn(*efficiency_map.shape)
    efficiency_map = np.clip(efficiency_map, 0.6, 0.95)
    
    # Initialize sequence processor
    processor = SequenceDataProcessor()
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Sequence Data Processing for RNN Training', fontsize=16, fontweight='bold')
    
    # Plot 1: Original 2D efficiency map
    ax1 = axes[0, 0]
    im1 = ax1.contourf(SPEED, TORQUE, efficiency_map, levels=15, cmap='viridis')
    ax1.set_xlabel('Speed (RPM)', fontweight='bold')
    ax1.set_ylabel('Torque (Nm)', fontweight='bold')
    ax1.set_title('Original 2D Efficiency Map', fontweight='bold')
    plt.colorbar(im1, ax=ax1, label='Efficiency')
    
    # Add operating region
    ax1.contour(SPEED, TORQUE, efficiency_map, levels=[0.8, 0.85, 0.9], 
               colors='white', linewidths=2)
    
    # Convert to sequences using different methods
    sequence_types = ['row_wise', 'column_wise', 'spiral']
    sequences_data = []
    
    for i, seq_type in enumerate(sequence_types):
        processor = SequenceDataProcessor(seq_type)
        sequence, speed_seq, torque_seq = processor.map_to_sequence(
            efficiency_map, SPEED, TORQUE)
        
        sequences_data.append({
            'type': seq_type,
            'sequence': sequence,
            'speed_seq': speed_seq,
            'torque_seq': torque_seq
        })
        
        print(f"\n🔄 {seq_type.replace('_', ' ').title()} sequence:")
        print(f"   Length: {len(sequence)} points")
        print(f"   Efficiency range: {np.min(sequence):.3f} - {np.max(sequence):.3f}")
        print(f"   Valid points: {np.sum(sequence > 0)}/{len(sequence)}")
    
    # Plot 2: Row-wise sequence visualization
    ax2 = axes[0, 1]
    row_data = sequences_data[0]
    ax2.plot(row_data['sequence'], 'b-', linewidth=2, alpha=0.7)
    ax2.set_xlabel('Sequence Index', fontweight='bold')
    ax2.set_ylabel('Efficiency', fontweight='bold')
    ax2.set_title('Row-wise Sequence', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0.5, 1.0)
    
    # Highlight high-efficiency regions
    high_eff_mask = row_data['sequence'] > 0.85
    ax2.scatter(np.where(high_eff_mask)[0], row_data['sequence'][high_eff_mask], 
               color='red', s=20, alpha=0.8, label='High efficiency (>85%)')
    ax2.legend()
    
    # Plot 3: Column-wise sequence visualization
    ax3 = axes[0, 2]
    col_data = sequences_data[1]
    ax3.plot(col_data['sequence'], 'g-', linewidth=2, alpha=0.7)
    ax3.set_xlabel('Sequence Index', fontweight='bold')
    ax3.set_ylabel('Efficiency', fontweight='bold')
    ax3.set_title('Column-wise Sequence', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0.5, 1.0)
    
    # Plot 4: Spiral sequence visualization
    ax4 = axes[1, 0]
    spiral_data = sequences_data[2]
    ax4.plot(spiral_data['sequence'], 'r-', linewidth=2, alpha=0.7)
    ax4.set_xlabel('Sequence Index', fontweight='bold')
    ax4.set_ylabel('Efficiency', fontweight='bold')
    ax4.set_title('Spiral Sequence (center-out)', fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(0.5, 1.0)
    
    # Plot 5: Sequence comparison
    ax5 = axes[1, 1]
    min_length = min(len(data['sequence']) for data in sequences_data)
    
    for i, data in enumerate(sequences_data):
        ax5.plot(data['sequence'][:min_length], label=data['type'].replace('_', ' ').title(), 
                linewidth=2, alpha=0.7)
    
    ax5.set_xlabel('Sequence Index', fontweight='bold')
    ax5.set_ylabel('Efficiency', fontweight='bold')
    ax5.set_title('Sequence Method Comparison', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    ax5.set_ylim(0.5, 1.0)
    
    # Plot 6: Input features demonstration
    ax6 = axes[1, 2]
    
    # Create input features for sample sequence
    sample_design = np.array([100, 50, 30, 10, 10, 80, 40, 20, 5, 15, 45, 1.5])
    
    sample_sequence = row_data['sequence'][:50]  # First 50 points
    sample_speed = row_data['speed_seq'][:50]
    sample_torque = row_data['torque_seq'][:50]
    
    processor = SequenceDataProcessor('row_wise')
    input_features = processor.create_input_features(sample_speed, sample_torque, sample_design)
    
    # Visualize first few features
    feature_names = ['Speed', 'Torque', 'Speed×Torque', 'Speed²', 'Torque²']
    
    for i in range(5):
        ax6.plot(input_features[:50, i], label=feature_names[i], linewidth=2, alpha=0.7)
    
    ax6.set_xlabel('Sequence Index', fontweight='bold')
    ax6.set_ylabel('Normalized Feature Value', fontweight='bold')
    ax6.set_title('Input Features (First 5)', fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Create input processing pipeline visualization
    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    # Pipeline stages
    pipeline_stages = [
        '2D Performance Map',
        'Sequence Conversion',
        'Feature Engineering',
        'Normalization',
        'RNN Input Ready'
    ]
    
    pipeline_descriptions = [
        'Efficiency/Power Factor\nSpeed × Torque Grid\nShape: (15, 20)',
        'Row-wise/Column-wise/Spiral\n1D Sequence\nVariable Length',
        'Speed, Torque, Control\nGeometry Parameters\n14 Features/Point',
        'Min-Max Scaling\n[0, 1] Range\nPreserve Relations',
        'Tensor Format\nBatch × Sequence × Features\nReady for Training'
    ]
    
    # Create pipeline visualization
    y_positions = np.arange(len(pipeline_stages))
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
    
    for i, (stage, desc, color) in enumerate(zip(pipeline_stages, pipeline_descriptions, colors)):
        # Stage box
        rect = plt.Rectangle((1, y_positions[i] - 0.4), 2.5, 0.8, 
                            facecolor=color, edgecolor='black', linewidth=2, alpha=0.7)
        ax.add_patch(rect)
        
        # Stage text
        ax.text(2.25, y_positions[i], stage, ha='center', va='center', 
                fontsize=12, fontweight='bold')
        
        # Description text
        ax.text(4.5, y_positions[i], desc, ha='left', va='center', 
                fontsize=10, bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))
        
        # Arrows
        if i < len(pipeline_stages) - 1:
            ax.arrow(3.75, y_positions[i], 0.5, 0, head_width=0.15, head_length=0.1, 
                    fc='black', ec='black')
    
    # Add data shape indicators
    shapes = ['(15, 20)', '(N,)', '(N, 14)', '(N, 14)', '(batch, N, 14)']
    for i, shape in enumerate(shapes):
        ax.text(8.5, y_positions[i], f'Shape: {shape}', ha='left', va='center', 
                fontsize=9, style='italic', color='blue', fontweight='bold')
    
    ax.set_xlim(0, 10)
    ax.set_ylim(-0.5, len(pipeline_stages) - 0.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('RNN Input Data Processing Pipeline', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print processing summary
    print("\n📋 Sequence Processing Summary:")
    print("=" * 40)
    print(f"📐 Original map shape: {efficiency_map.shape}")
    print(f"📏 Sequence length range: {min(len(data['sequence']) for data in sequences_data)} - {max(len(data['sequence']) for data in sequences_data)}")
    print(f"🎯 Input feature dimension: 14")
    print(f"📊 Target dimension: 1 (efficiency or power factor)")
    print(f"🔄 Variable sequence length handling: ✅")
    print(f"⚡ Normalization efficiency: {len(row_data['sequence'])} points processed")
    print(f"🧠 Ready for RNN training: ✅")

demonstrate_sequence_processing()

## Section 4: RNN Architecture Fundamentals

### Sequence Modeling for Performance Maps

Recurrent Neural Networks (RNNs) are specifically designed to handle sequential data and capture temporal dependencies. For performance map prediction, RNNs excel at:

- **Variable sequence lengths**: Different motor designs have different operating envelopes
- **Spatial relationships**: Capturing dependencies between adjacent operating points
- **Context preservation**: Maintaining geometric and excitation information across sequences
- **Parameter sharing**: Same weights applied to all sequence positions

### Gated Recurrent Units (GRU)

GRU cells are preferred over simple RNNs for performance map prediction:

#### GRU Equations

**Update Gate**:
$$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$$

**Reset Gate**:
$$r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$$

**Candidate Hidden State**:
$$\tilde{h}_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h)$$

**Final Hidden State**:
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

where:
- $x_t$ is the input at time step $t$
- $h_{t-1}$ is the previous hidden state
- $\sigma$ is the sigmoid function
- $\odot$ denotes element-wise multiplication

### Bidirectional Processing

**Bidirectional GRUs** process sequences in both directions:

$$\overrightarrow{h}_t = \text{GRU}_{\rightarrow}(x_t, \overrightarrow{h}_{t-1})$$
$$\overleftarrow{h}_t = \text{GRU}_{\leftarrow}(x_t, \overleftarrow{h}_{t+1})$$
$$h_t = \overrightarrow{h}_t \oplus \overleftarrow{h}_t$$

**Benefits for Performance Maps**:
- **Context from both sides**: Each point considers preceding and following operating conditions
- **Better spatial understanding**: Improved capture of efficiency contours and patterns
- **Enhanced accuracy**: Particularly beneficial for regions near operating boundaries

### Time-Distributed Layers

**TimeDistributed** layers apply the same operation to each time step:

$$y_t = W_{out} h_t + b_{out} \quad \forall t \in \{1, ..., T\}$$

This enables:
- **Sequence-to-sequence prediction**: Output sequence matches input sequence length
- **Parameter efficiency**: Same output weights for all time steps
- **Parallel processing**: All outputs computed simultaneously during inference

In [ ]:
# RNN Architecture Fundamentals Implementation
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

class GRUCell(nn.Module):
    """Custom GRU cell implementation for understanding"""
    
    def __init__(self, input_size, hidden_size):
        super(GRUCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Update gate parameters
        self.W_z = nn.Linear(input_size, hidden_size)
        self.U_z = nn.Linear(hidden_size, hidden_size)
        
        # Reset gate parameters
        self.W_r = nn.Linear(input_size, hidden_size)
        self.U_r = nn.Linear(hidden_size, hidden_size)
        
        # Candidate hidden state parameters
        self.W_h = nn.Linear(input_size, hidden_size)
        self.U_h = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x, h_prev):
        """Forward pass through GRU cell"""
        # Update gate
        z = torch.sigmoid(self.W_z(x) + self.U_z(h_prev))
        
        # Reset gate
        r = torch.sigmoid(self.W_r(x) + self.U_r(h_prev))
        
        # Candidate hidden state
        h_candidate = torch.tanh(self.W_h(x) + self.U_h(r * h_prev))
        
        # Final hidden state
        h_new = (1 - z) * h_prev + z * h_candidate
        
        return h_new, (z, r, h_candidate)

class PerformanceMapRNN(nn.Module):
    """RNN model for performance map prediction"""
    
    def __init__(self, input_dim=14, hidden_dim=128, output_dim=1, 
                 num_layers=2, bidirectional=True, dropout=0.0):
        super(PerformanceMapRNN, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        
        # Input processing layers
        self.input_dense = nn.Linear(input_dim, hidden_dim)
        self.input_activation = nn.Tanh()
        
        # RNN layers
        self.rnn = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        # Adjust hidden dimension for bidirectional processing
        rnn_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        # Output layers
        self.output_dense = nn.Sequential(
            nn.Linear(rnn_output_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        
        # Final activation (sigmoid for efficiency/power factor)
        self.output_activation = nn.Sigmoid()
        
    def forward(self, x, mask=None):
        """Forward pass through the RNN model"""
        # Input processing
        h = self.input_activation(self.input_dense(x))
        
        # RNN processing
        if mask is not None:
            # Pack padded sequence for efficiency
            packed = nn.utils.rnn.pack_padded_sequence(
                h, mask.sum(dim=1).cpu(), batch_first=True, enforce_sorted=False
            )
            packed_output, _ = self.rnn(packed)
            rnn_output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        else:
            rnn_output, _ = self.rnn(h)
        
        # Output processing
        output = self.output_dense(rnn_output)
        output = self.output_activation(output)
        
        return output

def demonstrate_rnn_fundamentals():
    """Demonstrate RNN fundamentals and architectures"""
    
    print("🧠 RNN Architecture Fundamentals Demonstration")
    print("=" * 60)
    
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Create sample data
    batch_size = 4
    seq_length = 20
    input_dim = 14
    
    # Generate sample input sequences
    x = torch.randn(batch_size, seq_length, input_dim)
    
    # Create mask for variable sequence lengths
    seq_lengths = torch.tensor([20, 15, 18, 12])
    mask = torch.arange(seq_length).expand(batch_size, seq_length) < seq_lengths.unsqueeze(1)
    
    print(f"📊 Input tensor shape: {x.shape}")
    print(f"📏 Sequence lengths: {seq_lengths.tolist()}")
    print(f"🎭 Mask shape: {mask.shape}")
    
    # Initialize models with different configurations
    models = {
        'Simple RNN': PerformanceMapRNN(
            input_dim=input_dim, hidden_dim=64, bidirectional=False, num_layers=1
        ),
        'Bidirectional RNN': PerformanceMapRNN(
            input_dim=input_dim, hidden_dim=64, bidirectional=True, num_layers=1
        ),
        'Deep RNN': PerformanceMapRNN(
            input_dim=input_dim, hidden_dim=64, bidirectional=True, num_layers=2
        ),
        'RNN with Dropout': PerformanceMapRNN(
            input_dim=input_dim, hidden_dim=64, bidirectional=True, num_layers=2, dropout=0.3
        )
    }
    
    # Create visualization
    fig = plt.figure(figsize=(20, 12))
    
    # Plot 1: GRU Cell Mechanics
    ax1 = plt.subplot(2, 4, 1)
    
    # Create custom GRU cell for demonstration
    gru_cell = GRUCell(input_dim, 64)
    
    # Process a single time step
    x_sample = x[0, 0:1, :]  # First sample, first time step
    h_prev = torch.zeros(1, 64)
    
    h_new, gate_values = gru_cell(x_sample, h_prev.squeeze())
    
    # Visualize gate activations
    gate_names = ['Update Gate', 'Reset Gate', 'Candidate State']
    gate_values_list = [gate_values[0].squeeze().detach().numpy(),
                        gate_values[1].squeeze().detach().numpy(),
                        gate_values[2].squeeze().detach().numpy()]
    
    # Create bar plot of gate distributions
    for i, (name, values) in enumerate(zip(gate_names, gate_values_list)):
        subset = values[:16]  # First 16 units for visualization
        x_pos = np.arange(len(subset)) + i * (len(subset) + 2)
        ax1.bar(x_pos, subset, alpha=0.7, label=name)
    
    ax1.set_xlabel('Hidden Unit Index', fontweight='bold')
    ax1.set_ylabel('Activation Value', fontweight='bold')
    ax1.set_title('GRU Gate Activations', fontweight='bold')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Model Comparison
    ax2 = plt.subplot(2, 4, 2)
    
    model_outputs = {}
    for name, model in models.items():
        with torch.no_grad():
            output = model(x)
            model_outputs[name] = output[0, :, 0].detach().numpy()  # First sample, efficiency output
    
    for i, (name, output) in enumerate(model_outputs.items()):
        ax2.plot(output[:20], label=name, linewidth=2, alpha=0.8)
    
    ax2.set_xlabel('Time Step', fontweight='bold')
    ax2.set_ylabel('Predicted Efficiency', fontweight='bold')
    ax2.set_title('Model Architecture Comparison', fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1)
    
    # Plot 3: Bidirectional vs Unidirectional
    ax3 = plt.subplot(2, 4, 3)
    
    # Compare bidirectional vs unidirectional processing
    simple_model = models['Simple RNN']
    bidirectional_model = models['Bidirectional RNN']
    
    with torch.no_grad():
        simple_output = simple_model(x)
        bidirectional_output = bidirectional_model(x)
    
    # Plot first sample outputs
    sample_idx = 0
    ax3.plot(simple_output[sample_idx, :, 0].detach().numpy(), 
            label='Unidirectional', linewidth=2, alpha=0.8)
    ax3.plot(bidirectional_output[sample_idx, :, 0].detach().numpy(), 
            label='Bidirectional', linewidth=2, alpha=0.8)
    
    ax3.set_xlabel('Time Step', fontweight='bold')
    ax3.set_ylabel('Predicted Efficiency', fontweight='bold')
    ax3.set_title('Bidirectional vs Unidirectional', fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 1)
    
    # Plot 4: Hidden State Evolution
    ax4 = plt.subplot(2, 4, 4)
    
    # Extract hidden states from RNN
    model = models['Bidirectional RNN']
    
    # Process input and capture intermediate states
    h = model.input_activation(model.input_dense(x))
    
    # Get hidden states from all layers
    all_hidden = []
    input_h = h
    for i, layer in enumerate(model.rnn._all_weights):
        if i % 4 == 0:  # Only process forward layers for simplicity
            layer_idx = i // 4
            if layer_idx < model.rnn.num_layers:
                h, _ = model.rnn._impl[h](input_h, None)  # This is a simplified approach
                all_hidden.append(h[0, :, :].detach().numpy())  # First sample
                input_h = h
    
    # Plot hidden state evolution for first few units
    hidden_sample = all_hidden[0] if all_hidden else h[0, :, :].detach().numpy()
    
    for unit_idx in range(min(5, hidden_sample.shape[1])):
        ax4.plot(hidden_sample[:, unit_idx], label=f'Unit {unit_idx+1}', alpha=0.7)
    
    ax4.set_xlabel('Time Step', fontweight='bold')
    ax4.set_ylabel('Hidden State Value', fontweight='bold')
    ax4.set_title('Hidden State Evolution', fontweight='bold')
    ax4.legend(fontsize=8)
    ax4.grid(True, alpha=0.3)
    
    # Plot 5: Parameter Count Comparison
    ax5 = plt.subplot(2, 4, 5)
    
    model_names = []
    param_counts = []
    
    for name, model in models.items():
        param_count = sum(p.numel() for p in model.parameters())
        model_names.append(name.replace('\n', ' '))
        param_counts.append(param_count)
    
    bars = ax5.bar(model_names, param_counts, alpha=0.7, 
                   color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
    
    ax5.set_ylabel('Number of Parameters', fontweight='bold')
    ax5.set_title('Model Complexity Comparison', fontweight='bold')
    ax5.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, count in zip(bars, param_counts):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(param_counts)*0.01,
                f'{count:,}', ha='center', va='bottom', fontweight='bold')
    
    ax5.grid(True, alpha=0.3, axis='y')
    
    # Plot 6: Memory vs Accuracy Trade-off
    ax6 = plt.subplot(2, 4, 6)
    
    # Simulate performance metrics (in practice, these would be measured)
    hidden_dims = [32, 64, 128, 256]
    memory_usage = [dim * 4 / 1024 for dim in hidden_dims]  # KB (approximate)
    accuracy = [0.82, 0.87, 0.91, 0.93]  # Simulated accuracy
    
    ax6_twin = ax6.twinx()
    
    line1 = ax6.plot(hidden_dims, memory_usage, 'bo-', linewidth=2, label='Memory Usage')
    line2 = ax6_twin.plot(hidden_dims, accuracy, 'rs-', linewidth=2, label='Accuracy')
    
    ax6.set_xlabel('Hidden Dimension', fontweight='bold')
    ax6.set_ylabel('Memory Usage (KB)', color='b', fontweight='bold')
    ax6_twin.set_ylabel('Accuracy', color='r', fontweight='bold')
    ax6.set_title('Memory vs Accuracy Trade-off', fontweight='bold')
    
    # Combine legends
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax6.legend(lines, labels, loc='upper left')
    
    ax6.grid(True, alpha=0.3)
    
    # Plot 7: Training Speed Comparison
    ax7 = plt.subplot(2, 4, 7)
    
    # Simulate training times (in practice, these would be measured)
    training_times = [0.5, 0.8, 1.2, 1.5]  # seconds per batch
    
    bars = ax7.bar(model_names, training_times, alpha=0.7,
                   color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
    
    ax7.set_ylabel('Training Time (s/batch)', fontweight='bold')
    ax7.set_title('Training Speed Comparison', fontweight='bold')
    ax7.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, time_val in zip(bars, training_times):
        ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(training_times)*0.01,
                f'{time_val:.1f}s', ha='center', va='bottom', fontweight='bold')
    
    ax7.grid(True, alpha=0.3, axis='y')
    
    # Plot 8: Sequence Length Handling
    ax8 = plt.subplot(2, 4, 8)
    
    # Demonstrate variable sequence length handling
    test_seq_lengths = [5, 10, 15, 20, 25]
    processing_times = []
    
    model = models['Bidirectional RNN']
    
    for seq_len in test_seq_lengths:
        test_x = torch.randn(1, seq_len, input_dim)
        
        import time
        start_time = time.time()
        with torch.no_grad():
            output = model(test_x)
        end_time = time.time()
        
        processing_times.append((end_time - start_time) * 1000)  # Convert to ms
    
    ax8.plot(test_seq_lengths, processing_times, 'go-', linewidth=2, markersize=8)
    ax8.set_xlabel('Sequence Length', fontweight='bold')
    ax8.set_ylabel('Processing Time (ms)', fontweight='bold')
    ax8.set_title('Sequence Length Scalability', fontweight='bold')
    ax8.grid(True, alpha=0.3)
    
    # Add annotations
    for i, (seq_len, proc_time) in enumerate(zip(test_seq_lengths, processing_times)):
        ax8.annotate(f'{proc_time:.1f}ms', (seq_len, proc_time), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    plt.suptitle('RNN Architecture Fundamentals and Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print model summary
    print("\n📋 RNN Model Summary:")
    print("=" * 40)
    
    for name, model in models.items():
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        print(f"\n🔧 {name}:")
        print(f"   Total parameters: {total_params:,}")
        print(f"   Trainable parameters: {trainable_params:,}")
        print(f"   Hidden dimension: {model.hidden_dim}")
        print(f"   Bidirectional: {model.bidirectional}")
        print(f"   Layers: {model.num_layers}")
    
    # Test model forward pass
    print("\n🚀 Forward Pass Test:")
    model = models['Bidirectional RNN']
    
    with torch.no_grad():
        output = model(x, mask)
        
    print(f"   Input shape: {x.shape}")
    print(f"   Output shape: {output.shape}")
    print(f"   Output range: [{output.min().item():.3f}, {output.max().item():.3f}]")
    print(f"   Mask applied: {mask is not None}")
    
    return models

# Run the demonstration
models = demonstrate_rnn_fundamentals()

This completes the first 3 sections of Chapter 3. We've established:

1. **Motivation and Background**: Climate change context, EV adoption, and computational challenges
2. **Electric Motor Fundamentals**: IPM vs FSCW motors, control strategies, performance characteristics
3. **Data Generation and Processing**: Latin Hypercube sampling, 4-stage FE process, sequence representation

### 🎯 Key Takeaways So Far:

- **Environmental Impact**: Transportation contributes 28% of U.S. emissions, with EVs offering a viable solution
- **Performance Maps**: Critical for EV motor design, traditionally requiring hours of FE computation
- **Motor Types**: IPM and FSCW motors offer different trade-offs for EV applications
- **Data Pipeline**: 4-stage FE process can be replaced by deep learning for massive speedup
- **Sequence Processing**: 2D maps converted to 1D sequences enable RNN-based prediction

### 🚀 Next Steps:

We'll now continue with the RNN architecture fundamentals and attention mechanisms that form the core of our performance map prediction approach.